# Collected football data: exploration, cleaning and preprocessing

This notebook audits the retained real archives across seasons and documents the project feature contract. It keeps FBref performance, Understat shot events, and Transfermarkt context separate. No new downloads, synthetic substitutes, model fitting, global imputation or scaling occur. The existing Streamlit model remains a separately labelled legacy baseline.

In [ ]:
from pathlib import Path
import os,sys,json
root=Path.cwd()
while not (root/'src/config.py').exists():
    if root==root.parent: raise RuntimeError('Run inside the existing repository')
    root=root.parent
os.chdir(root)
sys.path.insert(0,str(root))
os.environ.setdefault('MPLCONFIGDIR','/tmp/tfm-matplotlib')
import pandas as pd
from IPython.display import display,Markdown,Image
from src.config import ROOT,FEATURE_COLS
from scripts.collected_eda import run,load_verified_tables
summary=run()
manifest,tables=load_verified_tables()
players=tables['collected_player_seasons']
eligible=tables['collected_baseline_candidates']
assert not players.source.str.contains('demo|synthetic',case=False).any()
assert not eligible.partial_season.any()
assert not eligible.pos_main.eq('GK').any()
display(Markdown('**Dataset version:** '+manifest['dataset_version']+'\n\nRaw inputs and the current model are unchanged.'))

## Mandatory project features

The existing ten core per-90 features remain in the schema. Identity, club, season, league, position, age and minutes support scouting filters. Budget scenarios additionally require observed EUR values and both effective/known-at dates. Birth year does not establish DOB. xAG is not xA. Unknown values stay missing; schema completeness does not establish measured coverage. G+A is retained for reporting but excluded from model candidate features.

In [ ]:
display(pd.read_csv(ROOT/'reports/mandatory_features.csv'))
display(pd.read_csv(ROOT/'reports/collected_inventory.csv'))
display(pd.read_csv(ROOT/'reports/collected_season_summary.csv'))

## Coverage and missingness

Compare actual observed coverage rather than assuming every season has the same tables. Auxiliary totals with incompatible exposure were already nulled by the archive audit. Missing counts in one club stint propagate to the player-season total. Partial 2025/26 remains separate; no defensive values are backfilled from adjacent seasons.

In [ ]:
display(pd.read_csv(ROOT/'reports/collected_missingness.csv').query("table == 'collected_player_seasons' and missing > 0") [['field','missing','missing_fraction']])
display(Image(filename=str(ROOT/'reports/collected_feature_availability.png')))
display(Image(filename=str(ROOT/'reports/collected_season_coverage.png')))

## Duplicate handling, transfers between clubs and exposure

Club stints are preserved separately. Player-season rates use summed minutes and compatible totals; they are not averages of stint rates. Aggregation happens before the 450-minute floor, and multi-club records list every club without claiming which was current at a historical decision date. Exact duplicates are removed; conflicting FBref keys fail preprocessing.

In [ ]:
display(players.loc[players.club_count.gt(1),['player','season','clubs','minutes','gls','ast','gls_p90']].head(12))
display(pd.DataFrame([manifest['exact_duplicates_removed']]))
display(pd.read_csv(ROOT/'reports/collected_invalid_values.csv').query('invalid_values_nulled > 0'))
display(pd.read_csv(ROOT/'reports/collected_legacy_exclusions.csv'))

## Distributions and outliers

Inspect rates after controlling minimum exposure and player role. Extreme observed values are flagged, not clipped. The distributions below use development seasons only. Age is retained where the source provides it, but no missing DOB, height, foot or price is fabricated.

In [ ]:
display(eligible[['minutes','age']+FEATURE_COLS].describe().round(3))
display(pd.read_csv(ROOT/'reports/collected_outliers.csv'))
display(Image(filename=str(ROOT/'reports/collected_distributions.png')))
display(Markdown(f"**Finding:** {summary['outlier_flags']} player-season records carry review flags; {summary['outliers_in_historical_candidates']} enter the historical baseline candidate set. {summary['missing_ages']} ages remain unknown."))

## Correlations and redundant features

Spearman correlations use the shared 2020/21–2021/22 development period only, excluding both designs’ validation and holdout seasons. G+A is excluded because it is exactly goals plus assists; retaining it in a similarity feature vector would overweight attack output. Missing xA is excluded rather than substituted with xAG. Correlations are descriptive and do not establish causality or a validated feature-selection rule.

In [ ]:
display(pd.read_csv(ROOT/'reports/collected_training_correlations.csv',index_col=0).round(2))
display(Image(filename=str(ROOT/'reports/collected_training_correlations.png')))

## Position, season and league comparisons

Outfield comparisons separate DF/MF/FW. Goalkeepers remain in the inventory but require goalkeeper-specific metrics before goalkeeper recommendations are meaningful. Sparse 2024/25 defensive coverage makes those medians unrepresentative; inspect observed counts beside each estimate. A real La Liga–Segunda comparison is unavailable; synthetic Segunda rows are excluded.

In [ ]:
display(pd.read_csv(ROOT/'reports/collected_position_comparisons.csv'))
display(Image(filename=str(ROOT/'reports/collected_position_comparisons.png')))
styles=pd.read_csv(ROOT/'reports/collected_style_coverage.csv')
display(styles.groupby(['season','style'])[['rows','rows_above_60pct']].sum())
display(Markdown('Broad styles remain **statistical proxies**, even when their configured weights have adequate coverage. Direct pressing, transitions and goalkeeper metrics have not been collected.'))

## Understat shot-event cleaning

The archive changes from h_a to home_away for the shot side; the cleaner reconciles both and validates the values before assigning a club. Own goals are retained separately and excluded from attacking shots, goals and xG. Match counts here mean matches with recorded events, not player appearances. Unknown minutes prevent per-90 rates. Provider timestamps have unspecified timezone; they are not silently labelled UTC.

In [ ]:
display(pd.read_csv(ROOT/'reports/collected_understat_coverage.csv'))
display(tables['collected_understat_shooters'].head())
display(Markdown(f"**Finding:** {summary['understat_shot_records']:,} event records contain {summary['understat_attacking_shots']:,} attacking shots and {summary['understat_own_goals']} own-goal events."))

## Retained Transfermarkt context and identity review

DOB and height units are normalized for auditing, but profile attributes have no verified historical as-of date. Values remain separate from fees; missing values/fees are not zero, and an upstream zero fee is not automatically a verified free transfer. Arrival/departure records are retained as source records and grouped into potential movements without claiming uniquely dated transfers. Source-use flags remain attached. No new data are collected and these records do not enter a fitted model.

In [ ]:
display(pd.DataFrame([{k:summary[k] for k in ['transfermarkt_profiles','missing_market_values','parsed_dobs','parsed_heights','transfer_records','potential_movement_groups','unverified_zero_fees','missing_fees','eligible_historical_budget_records','segunda_scorer_sample_rows']}]))
queue=tables['identity_candidates']
display(queue.groupby(['right_source','status']).size().rename('pairs').reset_index())
display(queue.groupby('right_source').ambiguous.sum().rename('ambiguous_pairs').reset_index())
display(Markdown(f"**Finding:** {summary['candidate_identity_pairs']:,} identity proposals; {summary['reviewed_identity_pairs']} approved pairs. Name/club/season corroboration does not automatically join sources. {summary['identity_birth_year_conflicts']} upstream mappings conflict with reported birth years."))
display(tables['collected_segunda_scorer_sample'][['player','team','season','minutes','gls','ast','observation_scope']].head())
display(Markdown('The 25-row Segunda scorer sample is retained separately; it is biased toward goalscorers and does not support a full-league comparison or historical model cohort.'))

## Preprocessing decisions and next experiments

The offensive baseline uses train=2020/21–2022/23, validation=2023/24 and holdout=2024/25. The full-core proxy uses train=2020/21–2021/22, validation=2022/23 and holdout=2023/24, with 2024/25 only sparse monitoring. Both keep partial 2025/26 as monitoring. These distinct designs accommodate the missing 2024/25 defensive coverage. These season splits prevent directly pooling future seasons into development, but archive publication timestamps are insufficient for strict historical as-of validation. Fit imputation/scaling/PCA/K-means only inside a later training experiment. Do not use 2024/25 outcomes to tune features or weights. Offensive comparisons can use consistently observed fields; the full ten-feature tactical proxy has almost no eligible 2024/25 coverage. Budget and actual-transfer success validation remain unsupported.

In [ ]:
display(pd.DataFrame([summary]))
display(Markdown(f"**Prepared:** {summary['historical_baseline_candidates']:,} historical baseline candidates and {summary['historical_complete_core_candidates']:,} fully measured core-proxy candidates. This notebook reports preprocessing and EDA, **not model evaluation results**."))